In [3]:
###------Installed packages-----
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

###------Working Paper's functions------
from aml_working_paper.data.preprocess import preprocess_data
from aml_working_paper.config import RAW_DATA_DIR

### 1. Get the Synhetic Data

In [4]:
df_aml = pd.read_csv( "{}/synthetic_df_aml.csv".format(RAW_DATA_DIR))

### 2. Define the varaibles the model can observe

In [5]:
### Economic context and account exposure are not added since those two affect direcly he aml state
### But! they could be added as emmissions columns as another experiment
observed_columns = ["transaction_volume","cross_border_ratio","structuring_score"]

### 3. Spliting accounts into train, val and test sets

In [6]:
### The splitting is done by account because doing it in the normal way (just splitting the data by rows)
### would mix one account's period between train, valid and/or test which would introduce overfitting
account_ids = df_aml["account_id"].unique()

train_accounts, temp_accounts = train_test_split(account_ids, train_size=0.70, random_state=42)
val_accounts, test_accounts = train_test_split(temp_accounts, train_size=0.70, random_state=42)

train_df = df_aml[df_aml["account_id"].isin(train_accounts)].copy(deep=True)
val_df = df_aml[df_aml["account_id"].isin(val_accounts)].copy(deep=True)
test_df = df_aml[df_aml["account_id"].isin(test_accounts)].copy(deep=True)

##### Check the proportions of the aml states. 1 (suspicious) and 2 (alert) should be lower than 0 (normal)

In [9]:
print(train_df["aml_state"].value_counts(normalize=True).sort_index())

aml_state
0    0.482714
1    0.257000
2    0.260286
Name: proportion, dtype: float64


In [10]:
print(val_df["aml_state"].value_counts(normalize=True).sort_index())

aml_state
0    0.480476
1    0.238095
2    0.281429
Name: proportion, dtype: float64


In [11]:
print(test_df["aml_state"].value_counts(normalize=True).sort_index())

aml_state
0    0.444444
1    0.253333
2    0.302222
Name: proportion, dtype: float64


### 4. Apply standarization 

In [12]:
scaler = StandardScaler()

In [ ]:
###log1p is used because
###1. If there is 0 normal log can give inf values which then gives errors in training
###2. It helps reduce outliers.
### For future work, Standard scaler could be replaced with Robustscaler. 
transform_df_train = train_df.copy(deep=True)
features = transform_df_train[observed_columns].copy(deep=True)

features["transaction_volume"] = np.log1p(features["transaction_volume"])

transform_df_train[["transaction_volume_log","cross_border_ratio_scaled","structuring_score_scaled"]] = scaler.fit_transform(features)

In [ ]:
### Do the same for the val and test sets
transform_df_val = val_df.copy()
features = transform_df_val[observed_columns].copy()

features["transaction_volume"] = np.log1p(features["transaction_volume"])

transform_df_val[["transaction_volume_log","cross_border_ratio_scaled","structuring_score_scaled"]] = scaler.fit_transform(features)

In [ ]:
transform_df_test = test_df.copy()

features = transform_df_test[observed_columns].copy()
features["transaction_volume"] = np.log1p(features["transaction_volume"])
### not using fit_transform since I am using the same scaler from "fit_transform_train" 
scaled_features_test = scaler.transform(features) 
transform_df_test[["transaction_volume_log","cross_border_ratio_scaled","structuring_score_scaled"]] = scaler.fit_transform(features)

### 5. Build Sequences

In [ ]:
### convert the DataFrames into a collection of time series, one for each bank account.
### This is due to HMM process the evolution of the aml state through time and not as independent values.
sequences_train = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_train = transform_df_train.sort_values(["account_id", "time"])

for _, account_df in df_sorted_train.groupby("account_id", sort=True):
    ### obtain each account observed columns and transform it to a numpy matrix
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_train.append(sequence)

In [17]:
sequences_val = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_val = transform_df_val.sort_values(["account_id", "time"])

for _, account_df in df_sorted_val.groupby("account_id", sort=True):
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_val.append(sequence)

In [18]:
sequences_test = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_test = transform_df_test.sort_values(["account_id", "time"])

for _, account_df in df_sorted_test.groupby("account_id", sort=True):
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_test.append(sequence)

### 6. Wrap everything into preprocess_data

In [33]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)